In [1]:
# Use the regressor trained in 3A_2 to make full prediction dataframe
# convert modeled dataframe to one GeoTIFF per month

# Workflow
# 1. Load config, model, and spatial template
# 2. Iterate over months in the period range
# 3. Per month: load rows from parquet, predict, reconstruct DataArray
# 4. Align DataArray to template extent via reproject_match
# 5. Save as float32 GeoTIFF

# MAjor change: read parquets year by year for predictions

In [2]:
#libraries
import os
import glob
import joblib
import json
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
from tqdm import tqdm

#variables
START_YEAR,  START_MONTH = 2001, 1
END_YEAR,    END_MONTH   = 2005, 12

#paths
OUT_BASE = fr"..\Data_Temp\Processed\Monthly_downscaling"
model_path = fr"..\Data_Temp\Processed\Monthly_downscaling\Saved_regressors"
template = rxr.open_rasterio(r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_max.tif")

tmax_temp_dir = os.path.join(OUT_BASE, 'Temp_tmax')
tmax_yearly_parquet_dir = os.path.join(tmax_temp_dir, 'Tmax_yearly_parquet')
tmax_final_pred_dir = os.path.join(OUT_BASE, "Final_products", "Tmax")

tmin_temp_dir = os.path.join(OUT_BASE, 'Temp_tmin')
tmin_yearly_parquet_dir = os.path.join(tmin_temp_dir, 'Tmin_yearly_parquet')
tmin_final_pred_dir = os.path.join(OUT_BASE, "Final_products", "Tmin")

for d in [tmax_temp_dir, tmax_yearly_parquet_dir, tmax_final_pred_dir,
    tmin_temp_dir, tmin_yearly_parquet_dir, tmin_final_pred_dir]:
    if not os.path.exists(d):
        os.makedirs(d)
        
# read predictor features and model for each variable 
def get_model_features(t_var:str):
    metadata_file = glob.glob(os.path.join(model_path, f"{t_var}_*_feature_metadata.json"))[0]
    with open(metadata_file, 'r') as f:
            metadata = json.load(f)
    predictor_features = metadata['input_features']
    model_type = metadata['model_type']
    return model_type, predictor_features

model_type_tmax, predictor_features_tmax = get_model_features("tmax")
model_type_tmin, predictor_features_tmin = get_model_features("tmin")

In [3]:
# ── Main prediction loop ─────────────────────────────────────────────────────
# 
# For each month:
#   0. Load the model
#   1. Skip if the output raster already exists (safe to re-run)
#   2. Read only that month's rows from parquet (memory-efficient)
#   3. Predict with the RF model (using .values to suppress feature-name warning)
#   4. Pivot predictions back to a 2-D DataArray indexed by (y, x)
#   5. Write CRS / nodata / transform, then align to full template extent
#   6. Save as float32 GeoTIFF

month_range = pd.period_range(
    start = f"{START_YEAR}-{START_MONTH:02d}",
    end   = f"{END_YEAR}-{END_MONTH:02d}",
    freq  = "M",
)

for t_variable in ['tmax', 'tmin']:
    print(f"\n Predicting Monthly rasters for {t_variable}==========================================")
    if t_variable == 'tmax':
        _, predictor_features = model_type_tmax, predictor_features_tmax
        model = joblib.load(os.path.join(model_path, f'{t_variable}_{model_type_tmax}.pkl'))
    else:
        _, predictor_features = model_type_tmin, predictor_features_tmin
        model = joblib.load(os.path.join(model_path, f'{t_variable}_{model_type_tmax}.pkl'))

    for p in tqdm(month_range, desc="Predict rasters"):
        year, month = p.year, p.month
        if t_variable == "tmax":
            outpath = os.path.join(tmax_final_pred_dir, f"tmax_{year}{month:02d}.tif")
            df_year_path = os.path.join(tmax_yearly_parquet_dir, f"full_{year}.parquet")
        else:
            outpath = os.path.join(tmin_final_pred_dir, f"tmin_{year}{month:02d}.tif")
            df_year_path = os.path.join(tmin_yearly_parquet_dir, f"full_{year}.parquet")

        # Skip if already done 
        if os.path.exists(outpath):
            tqdm.write(f"  [{year}-{month:02d}] already exists, skipping.")
            continue

        # Load this month's rows from parquet 
        df_month = pd.read_parquet(
            df_year_path,
            filters=[("year", "==", year), ("month", "==", month)],
        )

        if df_month.empty:
            tqdm.write(f"  [{year}-{month:02d}] no rows found, skipping.")
            continue

        # Predict
        df_month[f"{t_variable}_predicted"] = model.predict(
            df_month[predictor_features]
        )

        # Make a rio xarray raster
        da = df_month.set_index(["y", "x"])[f"{t_variable}_predicted"].to_xarray()
        da = (
            da
            .rio.write_crs(template.rio.crs)
            .rio.write_nodata(-9999)
            .rio.write_transform(template.rio.transform())
        )
        da = da.rio.reproject_match(template)
        da.rio.to_raster(outpath, dtype="float32")

    print(f"\nDone. All rasters saved for {t_variable}")


 Predicting Monthly rasters for tmax==========================================


Predict rasters: 100%|██████████| 60/60 [04:24<00:00,  4.40s/it]



Done. All rasters saved for tmax

 Predicting Monthly rasters for tmin==========================================


Predict rasters: 100%|██████████| 60/60 [03:47<00:00,  3.79s/it]


Done. All rasters saved for tmin
